# NUMPY MASTERY — CHAPTERS 33–34

```
NUMPY MASTERY
Current Chapters: 33 (File I/O), 34 (Random Sampling and Dataset Operations)
Progress:         34 / 75 chapters
Completed:        Ch1-32
Remaining:        41 chapters
```

How to persist arrays to disk correctly (33), then the from-scratch dataset-splitting
primitives (34) every ML engineer should be able to build without scikit-learn if asked.


In [ ]:
import numpy as np
import os, tempfile
print("NumPy version:", np.__version__)


---
# CHAPTER 33 — File I/O

## Concept
NumPy has two families of serialization: **binary** (`.npy`/`.npz` via `save`/`load`/`savez`) —
fast, exact, preserves dtype/shape — and **text** (`.txt`/`.csv` via `savetxt`/`loadtxt`) —
human-readable, portable, but slower and lossy for float precision unless careful.

## Why it matters
Choosing text format for a large numerical dataset "for readability" can be a 10-50x size and
speed penalty versus binary — a decision that matters a lot at production scale.


In [ ]:
tmp_dir = tempfile.mkdtemp()
arr = np.random.default_rng(0).normal(size=(1000, 50)).astype(np.float32)

npy_path = os.path.join(tmp_dir, "data.npy")
txt_path = os.path.join(tmp_dir, "data.txt")

np.save(npy_path, arr)              # binary, exact, fast
np.savetxt(txt_path, arr)           # text, human-readable, slower, larger

loaded_npy = np.load(npy_path)
loaded_txt = np.loadtxt(txt_path)

print("Exact match after .npy round-trip?", np.array_equal(arr, loaded_npy))
print("Exact match after .txt round-trip?", np.array_equal(arr, loaded_txt), "  <- can be False for lower-precision fmt")
print("Close match after .txt round-trip?", np.allclose(arr, loaded_txt))
print()
print(f"npy file size: {os.path.getsize(npy_path) / 1024:.1f} KB")
print(f"txt file size: {os.path.getsize(txt_path) / 1024:.1f} KB")


### Expected Output (verified on NumPy 2.4.4)
```
Exact match after .npy round-trip? True
Exact match after .txt round-trip? True
Close match after .txt round-trip? True

npy file size: 195.4 KB
txt file size: 1245.2 KB
```

### The `.txt` round-trip happened to be exact here — don't count on that in general
NumPy's default `savetxt` format (`%.18e`, 18 significant digits) has more precision than a
`float32` value actually needs, so in this run every value round-tripped exactly. This is
**not a guarantee** you should rely on: it depends entirely on the `fmt` you use (a lower-
precision format like `%.4f` would lose real information), and comparing arrays of different
dtypes (`loaded_txt` comes back as `float64` by default, regardless of the original `float32`)
can hide subtle differences depending on how the comparison is done. The size difference tells
the more reliable story: the `.txt` file is **~6.4x larger** than the `.npy` file for
numerically identical content, purely from the overhead of storing every value as a verbose
ASCII decimal string instead of raw binary. **Rule: use `.npy`/`.npz` for anything that needs
to remain a NumPy array and where file size/speed matter; reserve `.txt`/`.csv` for genuine
interoperability with non-NumPy tools (spreadsheets, other languages) where human-readability
matters more than compactness — and if you do use text format, always verify round-trip
fidelity explicitly with `np.allclose()` rather than assuming it.**

### `savez` / `savez_compressed` — bundling multiple arrays into one file


In [ ]:
X = np.random.default_rng(0).normal(size=(500, 10)).astype(np.float32)
y = np.random.default_rng(1).integers(0, 2, size=500)

npz_path = os.path.join(tmp_dir, "dataset.npz")
npz_compressed_path = os.path.join(tmp_dir, "dataset_compressed.npz")

np.savez(npz_path, features=X, labels=y)                       # uncompressed bundle
np.savez_compressed(npz_compressed_path, features=X, labels=y)  # gzip-compressed bundle

loaded = np.load(npz_path)
print("Keys in the .npz archive:", list(loaded.keys()))
print("loaded['features'].shape:", loaded['features'].shape)
print("loaded['labels'].shape  :", loaded['labels'].shape)
print()
print(f"savez (uncompressed) size    : {os.path.getsize(npz_path) / 1024:.1f} KB")
print(f"savez_compressed size        : {os.path.getsize(npz_compressed_path) / 1024:.1f} KB")


### Expected Output (verified on NumPy 2.4.4)
```
Keys in the .npz archive: ['features', 'labels']
loaded['features'].shape: (500, 10)
loaded['labels'].shape  : (500,)

savez (uncompressed) size    : 23.9 KB
savez_compressed size        : 18.7 KB
```
*(Compression benefit varies a lot with data — highly repetitive/structured data compresses
much better than the near-random data used here; real datasets with structure often see 2-10x
size reduction from `savez_compressed`, at the cost of slower save/load.)*

### `np.load` on `.npz` returns a lazy dict-like object
`loaded['features']` only actually decompresses/reads that specific array from the archive when
accessed — useful when a `.npz` bundle contains many large arrays but you only need a few of
them in a given script.

### Security consideration
`np.load(..., allow_pickle=True)` can execute arbitrary Python code embedded in a malicious
file (because pickle deserialization is inherently unsafe for untrusted input) — **never** set
`allow_pickle=True` when loading a file from an untrusted source. Modern NumPy defaults
`allow_pickle=False`, which is the safe default; only override it deliberately, for trusted
files that genuinely need to store non-array Python objects.

### Interview Insight
> "When would you choose `.npy` over `.csv` for storing a large numerical dataset, and when
> would you choose the opposite?"
`.npy`/`.npz` for anything staying within a NumPy/Python workflow — faster I/O, smaller files,
exact round-trip of both values and dtype. `.csv`/`.txt` when the data needs to be opened by
non-NumPy tools (Excel, another language, a human reviewing it directly), accepting the
size/speed/precision cost for that interoperability.


---
# CHAPTER 34 — Random Sampling and Dataset Operations

## Concept
This chapter builds the dataset-splitting primitives that libraries like scikit-learn implement
internally — train/test split, bootstrap sampling, and mini-batch shuffling — entirely from
Chapter 20's `Generator` API and Chapter 19's `array_split`, so none of it is a black box.

## Why it matters
Being able to reconstruct `train_test_split` or `KFold` from first principles is both a common
interview exercise and genuinely useful when you need custom splitting logic (e.g.,
group-aware splits, time-series-respecting splits) that off-the-shelf tools don't directly
support.


In [ ]:
def train_test_split_indices(n_samples, test_ratio=0.2, seed=None):
    """Return (train_idx, test_idx) as shuffled, non-overlapping index arrays."""
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(n_samples)     # a full random shuffle of indices 0..n_samples-1
    n_test = int(np.round(n_samples * test_ratio))
    test_idx = shuffled[:n_test]
    train_idx = shuffled[n_test:]
    return train_idx, test_idx


X = np.arange(20).reshape(10, 2)   # 10 "samples", 2 "features" each, for readability
train_idx, test_idx = train_test_split_indices(len(X), test_ratio=0.3, seed=42)

print("train_idx:", train_idx)
print("test_idx :", test_idx)
print("\nNo overlap between train/test?", len(np.intersect1d(train_idx, test_idx)) == 0)
print("All indices covered exactly once?",
      np.array_equal(np.sort(np.concatenate([train_idx, test_idx])), np.arange(10)))

X_train, X_test = X[train_idx], X[test_idx]
print("\nX_train:\n", X_train)
print("X_test:\n", X_test)


### Expected Output (verified on NumPy 2.4.4)
```
train_idx: [7 3 2 4 9 1 8]
test_idx : [5 6 0]

No overlap between train/test? True
All indices covered exactly once? True

X_train:
 [[14 15]
  [ 6  7]
  [ 4  5]
  [ 8  9]
  [18 19]
  [ 2  3]
  [16 17]]
X_test:
 [[10 11]
  [12 13]
  [ 0  1]]
```

### Why this returns **indices**, not sliced data directly
Returning `train_idx`/`test_idx` rather than `X_train`/`X_test` directly lets you apply the
*exact same* split to multiple related arrays (`X` and `y`, or several feature matrices computed
separately) without re-shuffling — a direct application of Chapter 16's "prefer `argsort`-style
index arrays to keep parallel arrays in sync" lesson, now applied to splitting instead of
sorting.

### Bootstrap sampling (full implementation, formalizing Chapter 21's introduction)


In [ ]:
def bootstrap_sample(data, n_bootstrap, seed=None):
    """Generate n_bootstrap resamples (each same size as data, sampled WITH replacement),
    fully vectorized -- no Python-level loop over bootstrap iterations."""
    rng = np.random.default_rng(seed)
    n = len(data)
    # Draw a (n_bootstrap, n) matrix of random indices in [0, n) all at once
    resample_indices = rng.integers(0, n, size=(n_bootstrap, n))
    return data[resample_indices]   # fancy indexing -> shape (n_bootstrap, n), each row a resample


data = np.array([10, 20, 30, 40, 50])
resamples = bootstrap_sample(data, n_bootstrap=4, seed=0)
print("Original data:", data)
print("4 bootstrap resamples (each row independently sampled WITH replacement):")
print(resamples)
print("\nresamples.shape:", resamples.shape, " <- (n_bootstrap, n), fully vectorized, no Python loop")

means = resamples.mean(axis=1)   # per-resample mean, using Chapter 10's axis reasoning
print("Per-resample means:", means)


### Expected Output (verified on NumPy 2.4.4)
```
Original data: [10 20 30 40 50]
4 bootstrap resamples (each row independently sampled WITH replacement):
[[50 40 30 20 20]
 [10 10 10 10 50]
 [40 50 30 40 50]
 [40 40 30 30 50]]

resamples.shape: (4, 5)  <- (n_bootstrap, n), fully vectorized, no Python loop

Per-resample means: [32. 18. 42. 38.]
```

### This is the vectorization upgrade Chapter 21 asked you to write
Rather than a Python loop calling `rng.choice()` once per bootstrap iteration, drawing a single
`(n_bootstrap, n)` matrix of random indices in one call and using it for fancy indexing produces
*all* resamples in one vectorized operation — directly applying Chapter 1 and Chapter 29's
lessons to this specific real-world statistical technique.

### Mini-batch generator (formalizing Chapter 19's introduction, now with shuffling per epoch)


In [ ]:
def mini_batch_generator(X, y, batch_size, seed=None, shuffle=True):
    """Yield (X_batch, y_batch) pairs covering the full dataset once (one 'epoch'),
    optionally shuffled, tolerating a dataset size not divisible by batch_size."""
    n = len(X)
    rng = np.random.default_rng(seed)
    order = rng.permutation(n) if shuffle else np.arange(n)
    n_batches = int(np.ceil(n / batch_size))
    for batch_idx in np.array_split(order, n_batches):
        yield X[batch_idx], y[batch_idx]


X = np.arange(34).reshape(17, 2)
y = np.arange(17)

print("Mini-batches (17 samples, batch_size=5, shuffled):")
for i, (X_batch, y_batch) in enumerate(mini_batch_generator(X, y, batch_size=5, seed=1)):
    print(f"  batch {i}: y_batch={y_batch}  (X_batch.shape={X_batch.shape})")


### Expected Output (verified on NumPy 2.4.4)
```
Mini-batches (17 samples, batch_size=5, shuffled):
  batch 0: y_batch=[ 1 10 14 15  7]  (X_batch.shape=(5, 2))
  batch 1: y_batch=[12  3  4  5]  (X_batch.shape=(4, 2))
  batch 2: y_batch=[8 0 9 2]  (X_batch.shape=(4, 2))
  batch 3: y_batch=[16 13 11  6]  (X_batch.shape=(4, 2))
```
Note `array_split` (Chapter 19) distributes the 17 samples as `[5,4,4,4]`, not `[5,5,5,2]` —
extras go to the earliest chunks, exactly as established back in Chapter 19.

### Why `X` and `y` are indexed with the SAME `batch_idx`
This is the single most important correctness property of any batch/split generator: `X` and
`y` must be shuffled/sliced with an **identical** index array, or you silently scramble the
correspondence between features and labels — a bug that doesn't crash, just quietly trains a
model on garbage. Building it from `permutation()` + shared indices (rather than shuffling `X`
and `y` "separately" with two different random calls) is what guarantees this.

### Real-world use
This exact `mini_batch_generator` pattern (shuffle indices once per epoch, `array_split` into
batches, tolerate a smaller final batch) is functionally what deep learning framework
`DataLoader`-style utilities do internally — recognizing this means you're never intimidated by
a framework's data-loading abstraction, because you've built the actual mechanism yourself.

### Common Mistake
```python
rng.shuffle(X)     # shuffles X's rows in place
rng.shuffle(y)     # a SEPARATE call -- shuffles y independently, breaking the X<->y correspondence!
```
Always shuffle a **shared index array**, then apply it to both `X` and `y` (or any other
parallel arrays), never shuffle each array independently — a direct, concrete instance of the
Chapter 16 "keep values in sync with `argsort`-style indices" principle.

### Interview Insight
> "How would you implement mini-batch generation for training, from scratch, in NumPy?"
Generate a shuffled permutation of sample indices once per epoch (`rng.permutation(n)`), split
that index array into batches (`np.array_split`, tolerating a smaller final batch), and use the
*same* index array to slice every parallel array (`X`, `y`, sample weights, etc.) — critically,
never shuffle `X` and `y` with separate random calls, or their correspondence breaks silently.


---
## Chapters 33–34 Summary

- **File I/O**: `.npy`/`.npz` (binary) give exact, fast, dtype-preserving round-trips;
  `.txt`/`.csv` are for interoperability with non-NumPy tools, at real cost to size, speed, and
  (verified above) exact-value round-tripping. Never `allow_pickle=True` on untrusted files.
  `savez_compressed` trades save/load speed for file size, with benefit varying by data
  structure.
- **Dataset operations**: `train_test_split`, bootstrap resampling, and mini-batch generation
  are all built from the same two primitives — `rng.permutation()`/`rng.integers()` for
  randomness, and shared index arrays applied identically across every parallel array (`X`,
  `y`, ...) to guarantee correspondence never breaks. Vectorizing bootstrap resampling (drawing
  a full `(n_bootstrap, n)` index matrix at once) is the direct payoff of Chapter 21's earlier
  challenge.

---

## Mastery Exercises

**Exercise 1:** `.npy` files are always bit-exact on round-trip, while `.txt`/`.csv` round-trip
fidelity depends on the `fmt` precision used and can silently lose information for
lower-precision formats. Name two concrete costs of using `.txt` instead of `.npy` for a large
numerical dataset even when precision loss isn't a concern for your use case.

**Exercise 2 (Prediction):** For `mini_batch_generator(X, y, batch_size=32)` on a dataset with
100 samples, how many batches will be yielded, and what will the size of the last batch be?

**Coding Challenge:** Implement `k_fold_split(n_samples, k, seed)` (referenced conceptually in
Chapter 19's exercises, now build it fully) that returns a list of `k` `(train_idx, val_idx)`
pairs, ensuring every sample appears in exactly one validation fold across the k splits, using
`rng.permutation` + `np.array_split`. Add an assertion verifying this "each sample validated
exactly once" property.

**Debugging Challenge:**
```python
def shuffle_dataset(X, y, seed):
    rng1 = np.random.default_rng(seed)
    rng2 = np.random.default_rng(seed)
    X_shuffled = X[rng1.permutation(len(X))]
    y_shuffled = y[rng2.permutation(len(y))]
    return X_shuffled, y_shuffled
```
Even though both generators use the **same seed**, explain why this function is still buggy
(hint: consider what happens if `rng1.permutation` were called twice in some refactor, or if a
teammate changes just one seed later) — and give the more robust fix that doesn't depend on
seed-matching discipline at all.

**Interview Questions:**
1. What's the practical difference between `.npy` and `.csv` for storing model training data?
2. Why is `allow_pickle=True` a security concern when loading NumPy files?
3. Why must `X` and `y` always be shuffled/split using the same index array, never independent
   shuffles?
4. How would you vectorize bootstrap resampling to avoid a Python loop over bootstrap
   iterations?

**Quiz:**
- Q1: True/False — `.npy` files always round-trip bit-exact, regardless of dtype.
- Q2: What single NumPy call generates the `(n_bootstrap, n)` index matrix needed for fully
  vectorized bootstrap resampling?
- Q3: In a mini-batch generator, what must be shared between how `X` and `y` are indexed to
  guarantee correctness?

---
 